In [1]:
import pandas as pd
import numpy as np
import simfin as sf

In [2]:
df = pd.read_csv("/Users/yashwanthkumar/Desktop/Desktop/ML project/early_layoff_warning/data/layoffs_cleaned.csv")
df.head(10)

,Company,Location_HQ,Industry,Laid_Off_Count,Date,Source,Funds_Raised,Stage,Date_Added,Country,Percentage,List_of_Employees_Laid_Off,Laid_Off_Count_Log
0,Oda,Oslo,Food,150.0,2024-06-05,https://techcrunch.com/2024/06/05/softbank-bac...,691.0,Unknown,2024-06-05 18:01:25,Norway,NaN,Unknown,5.017280
1,Pagaya,Tel Aviv,Finance,100.0,2024-06-05,https://www.calcalistech.com/ctechnews/article...,2000.0,Post-IPO,2024-06-05 23:11:24,Israel,0.20,Unknown,4.615121
2,Aleph Farms,Tel Aviv,Food,30.0,2024-06-05,https://www.calcalistech.com/ctechnews/article...,119.0,Unknown,2024-06-05 23:13:43,Israel,0.30,Unknown,3.433987
3,MoonPay,Dover,Crypto,30.0,2024-06-05,https://www.theblock.co/post/298638/moonpay-la...,651.0,Unknown,2024-06-05 23:12:47,United States,0.10,Unknown,3.433987
4,Microsoft,Seattle,Other,1000.0,2024-06-03,https://www.theverge.com/2024/6/3/24170902/mic...,1.0,Post-IPO,2024-06-03 20:27:12,United States,NaN,Unknown,6.908755
5,OrCam,Jerusalem,Healthcare,100.0,2024-06-03,https://www.calcalistech.com/ctechnews/article...,86.0,Unknown,2024-06-04 03:47:34,Israel,0.50,Unknown,4.615121
6,Google,SF Bay Area,Consumer,100.0,2024-05-31,https://www.businessinsider.com/google-cloud-l...,26.0,Post-IPO,2024-06-01 18:35:17,United States,NaN,Unknown,4.615121
7,Tropic,New York City,Finance,40.0,2024-05-31,https://www.linkedin.com/feed/update/urn:li:ac...,67.0,Series B,2024-06-01 18:33:50,United States,NaN,Unknown,3.713572
8,FlightStats,Portland,Travel,73.0,2024-05-30,https://www.oregonlive.com/silicon-forest/2024...,3.0,Acquired,2024-05-31 10:51:22,United States,NaN,Unknown,4.304065
9,ICANN,Los Angeles,Infrastructure,33.0,2024-05-30,https://www.icann.org/en/blogs/details/organiz...,172.5,Unknown,2024-06-05 18:07:23,United States,0.07,Unknown,3.526361


In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv('SIMFIN_API_KEY')
print(f"Key loaded: {api_key[:4]}...")

sf.set_api_key(api_key)
sf.set_data_dir('data/simfin/')

# Test - load quarterly income statements
df_income = sf.load_income(variant='quarterly', market='us')
print(df_income.shape)
print(df_income.head())

Key loaded: cd0a...
Dataset "us-income-quarterly" on disk (0 days old).
- Loading from disk ... Done!
(47760, 26)
                    SimFinId Currency  Fiscal Year Fiscal Period Publish Date  \
Ticker Report Date                                                              
A      2020-07-31      45846      USD         2020            Q3   2020-09-01   
       2020-10-31      45846      USD         2020            Q4   2020-12-18   
       2021-01-31      45846      USD         2021            Q1   2021-03-02   
       2021-04-30      45846      USD         2021            Q2   2021-06-01   
       2021-07-31      45846      USD         2021            Q3   2021-09-01   

                   Restated Date  Shares (Basic)  Shares (Diluted)  \
Ticker Report Date                                                   
A      2020-07-31     2021-09-01     309000000.0       312000000.0   
       2020-10-31     2021-09-01     308000000.0       311000000.0   
       2021-01-31     2022-03-03     3

In [4]:
df_balance = sf.load_balance(variant='quarterly', market='us')
print(df_balance.shape)
print(df_balance.columns.tolist())

Dataset "us-balance-quarterly" not on disk.
- Downloading ... 100.0%
- Extracting zip-file ... Done!
- Loading from disk ... Done!
(47763, 28)
['SimFinId', 'Currency', 'Fiscal Year', 'Fiscal Period', 'Publish Date', 'Restated Date', 'Shares (Basic)', 'Shares (Diluted)', 'Cash, Cash Equivalents & Short Term Investments', 'Accounts & Notes Receivable', 'Inventories', 'Total Current Assets', 'Property, Plant & Equipment, Net', 'Long Term Investments & Receivables', 'Other Long Term Assets', 'Total Noncurrent Assets', 'Total Assets', 'Payables & Accruals', 'Short Term Debt', 'Total Current Liabilities', 'Long Term Debt', 'Total Noncurrent Liabilities', 'Total Liabilities', 'Share Capital & Additional Paid-In Capital', 'Treasury Stock', 'Retained Earnings', 'Total Equity', 'Total Liabilities & Equity']


In [8]:
# Reset index to make Ticker and Report Date regular columns
df_income_clean = df_income[['Revenue', 'Net Income']].copy().reset_index()
df_balance_clean = df_balance[['Cash, Cash Equivalents & Short Term Investments']].copy().reset_index()

# Rename for clarity
df_balance_clean = df_balance_clean.rename(columns={
    'Cash, Cash Equivalents & Short Term Investments': 'Cash_Reserves'
})

# Merge on Ticker + Report Date
df_simfin = df_income_clean.merge(df_balance_clean, on=['Ticker', 'Report Date'], how='left')

print(df_simfin.shape)
print(df_simfin.head())

(47760, 5)
  Ticker Report Date       Revenue  Net Income  Cash_Reserves
0      A  2020-07-31  1.261000e+09   199000000   1.358000e+09
1      A  2020-10-31  1.483000e+09   222000000   1.441000e+09
2      A  2021-01-31  1.548000e+09   288000000   1.329000e+09
3      A  2021-04-30  1.525000e+09   216000000   1.380000e+09
4      A  2021-07-31  1.586000e+09   264000000   1.428000e+09


In [14]:
df_simfin = df_simfin.sort_values(["Ticker", "Report Date"])

df_simfin["Profit_Margin"] = df_simfin["Net Income"] / df_simfin["Revenue"]

df_simfin["Revenue_Growth"] = df_simfin.groupby("Ticker")["Revenue"].pct_change()

df_simfin = df_simfin.drop(columns=['Revenue', 'Net Income'])


print(df_simfin.shape)
print(df_simfin.head(10))
print(df_simfin.isnull().sum())



(47760, 5)
  Ticker Report Date  Cash_Reserves  Profit_Margin  Revenue_Growth
0      A  2020-07-31   1.358000e+09       0.157811             NaN
1      A  2020-10-31   1.441000e+09       0.149697        0.176051
2      A  2021-01-31   1.329000e+09       0.186047        0.043830
3      A  2021-04-30   1.380000e+09       0.141639       -0.014858
4      A  2021-07-31   1.428000e+09       0.166456        0.040000
5      A  2021-10-31   1.575000e+09       0.266265        0.046658
6      A  2022-01-31   1.158000e+09       0.169056        0.008434
7      A  2022-04-30   1.207000e+09       0.170504       -0.040024
8      A  2022-07-31   1.053000e+09       0.191502        0.069073
9      A  2022-10-31   1.053000e+09       0.199027        0.076251
Ticker               0
Report Date          0
Cash_Reserves      210
Profit_Margin     5218
Revenue_Growth    8698
dtype: int64


In [18]:
df["Date"] = pd.to_datetime(df["Date"])

df_simfin["Report Date"] = pd.to_datetime(df_simfin["Report Date"])

mapping_df = pd.read_csv("/Users/yashwanthkumar/Desktop/Desktop/ML project/early_layoff_warning/data/ticker_mapping.csv")

df = df.merge(mapping_df, on="Company", how="left")

print(f"Rows with ticker: {df['Ticker'].notna().sum()}")
print(f"Rows without ticker: {df['Ticker'].isna().sum()}")


Rows with ticker: 480
Rows without ticker: 1909


In [19]:
def get_financials_before_layoff(row, df_fin):
    # Private companies - no ticker
    if pd.isna(row['Ticker']):
        return pd.Series({
            'Cash_Reserves': np.nan,
            'Profit_Margin': np.nan,
            'Revenue_Growth': np.nan
        })
    
    # Get all quarters for this ticker
    ticker_data = df_fin[df_fin['Ticker'] == row['Ticker']]
    
    # Find quarters strictly before layoff date
    before = ticker_data[ticker_data['Report Date'] < row['Date']]
    
    if before.empty:
        return pd.Series({
            'Cash_Reserves': np.nan,
            'Profit_Margin': np.nan,
            'Revenue_Growth': np.nan
        })
    
    # Take the most recent quarter before layoff
    closest = before.sort_values('Report Date').iloc[-1]
    
    return pd.Series({
        'Cash_Reserves': closest['Cash_Reserves'],
        'Profit_Margin': closest['Profit_Margin'],
        'Revenue_Growth': closest['Revenue_Growth']
    })

# Apply - takes 1-2 minutes
print("Merging financials... takes a minute")
financial_cols = df.apply(
    lambda row: get_financials_before_layoff(row, df_simfin),
    axis=1
)

df = pd.concat([df, financial_cols], axis=1)

print(f"Shape: {df.shape}")
print(df[['Company', 'Ticker', 'Date', 'Cash_Reserves', 'Profit_Margin', 'Revenue_Growth']].head(10))

Merging financials... takes a minute
Shape: (2389, 19)
       Company Ticker       Date  Cash_Reserves  Profit_Margin  Revenue_Growth
0          Oda    NaN 2024-06-05            NaN            NaN             NaN
1       Pagaya    NaN 2024-06-05            NaN            NaN             NaN
2  Aleph Farms    NaN 2024-06-05            NaN            NaN             NaN
3      MoonPay    NaN 2024-06-05            NaN            NaN             NaN
4    Microsoft   MSFT 2024-06-03   8.002100e+10       0.354667       -0.002612
5        OrCam    NaN 2024-06-03            NaN            NaN             NaN
6       Google   GOOG 2024-05-31   1.080900e+11       0.293796       -0.066864
7       Tropic    NaN 2024-05-31            NaN            NaN             NaN
8  FlightStats    NaN 2024-05-30            NaN            NaN             NaN
9        ICANN    NaN 2024-05-30            NaN            NaN             NaN


In [20]:
print(f"Rows with Cash_Reserves: {df['Cash_Reserves'].notna().sum()}")
print(f"Rows with Profit_Margin: {df['Profit_Margin'].notna().sum()}")
print(f"Rows with Revenue_Growth: {df['Revenue_Growth'].notna().sum()}")
print(f"Total rows: {df.shape[0]}")

Rows with Cash_Reserves: 262
Rows with Profit_Margin: 259
Rows with Revenue_Growth: 255
Total rows: 2389


In [21]:
df['Laid_Off'] = 1
print(df['Laid_Off'].value_counts())

Laid_Off
1    2389
Name: count, dtype: int64


In [22]:
# S&P 500 companies that never appeared in our layoffs dataset
# These are your negative examples
sp500_never_laid_off = [
    'JNJ', 'PG', 'KO', 'PEP', 'WMT', 'MCD', 'MMM', 'ABT',
    'ACN', 'ADBE', 'ADP', 'AIG', 'ALL', 'AMGN', 'AXP', 'BA',
    'BAC', 'BK', 'BLK', 'BMY', 'BRK-B', 'C', 'CAT', 'CL',
    'CMCSA', 'COF', 'COP', 'COST', 'CVS', 'CVX', 'D', 'DE',
    'DIS', 'DOW', 'DUK', 'EMR', 'EXC', 'F', 'FDX', 'GD',
    'GE', 'GIS', 'GL', 'GM', 'GS', 'HAL', 'HD', 'HON',
    'HUM', 'IBM', 'ICE', 'INTC', 'JCI', 'JPM', 'K', 'KEY',
    'KMB', 'KR', 'L', 'LLY', 'LMT', 'LOW', 'MA', 'MET',
    'MGM', 'MO', 'MRK', 'MS', 'MSI', 'NEE', 'NEM', 'NKE',
    'NOC', 'NSC', 'NTRS', 'NUE', 'OKE', 'OMC', 'ORCL', 'OXY',
    'PAYX', 'PFE', 'PGR', 'PNC', 'PPG', 'PRU', 'PSA', 'PSX',
    'RTX', 'SBUX', 'SHW', 'SO', 'SPG', 'SYK', 'SYY', 'T',
    'TGT', 'TJX', 'TMO', 'TROW', 'TRV', 'TSN', 'TXN', 'UNH',
    'UNP', 'UPS', 'USB', 'V', 'VLO', 'VMC', 'VZ', 'WBA',
    'WFC', 'WHR', 'WM', 'XOM', 'YUM', 'ZBH'
]

# Remove any that actually appear in our layoffs dataset
layoff_tickers = df['Ticker'].dropna().unique().tolist()
clean_sp500 = [t for t in sp500_never_laid_off if t not in layoff_tickers]

print(f"S&P 500 non-layoff companies: {len(clean_sp500)}")

S&P 500 non-layoff companies: 113


In [23]:
snp500_financials = df_simfin[df_simfin["Ticker"].isin(clean_sp500)].copy()

print(snp500_financials.shape)
print(snp500_financials.nunique())
print(snp500_financials.head(10))

(1346, 5)
Ticker              78
Report Date         59
Cash_Reserves     1293
Profit_Margin     1346
Revenue_Growth    1268
dtype: int64
    Ticker Report Date  Cash_Reserves  Profit_Margin  Revenue_Growth
325    ABT  2020-09-30   4.731000e+09       0.139162             NaN
326    ABT  2020-12-31   7.148000e+09       0.202037        0.208743
327    ABT  2021-03-31   8.372000e+09       0.171480       -0.022895
328    ABT  2021-06-30   8.944000e+09       0.116306       -0.022284
329    ABT  2021-09-30   9.692000e+09       0.192167        0.068962
330    ABT  2021-12-31   1.024900e+10       0.173439        0.049414
331    ABT  2022-03-31   8.158000e+09       0.205717        0.037234
332    ABT  2022-06-30   9.290000e+09       0.179266       -0.053636
333    ABT  2022-09-30   9.907000e+09       0.137848       -0.075242
334    ABT  2022-12-31   1.017000e+10       0.102368       -0.030644


In [24]:
# Each quarter for these companies = a non-layoff observation = label 0
snp500_financials['Laid_Off'] = 0

# Add placeholder columns to match df structure
snp500_financials['Company'] = snp500_financials['Ticker']
snp500_financials['Industry'] = 'Unknown'
snp500_financials['Stage'] = 'Post-IPO'
snp500_financials['Funds_Raised'] = np.nan
snp500_financials['Country'] = 'United States'
snp500_financials['Location_HQ'] = 'Unknown'
snp500_financials['Laid_Off_Count'] = 0
snp500_financials['Laid_Off_Count_Log'] = 0
snp500_financials['Percentage'] = 0
snp500_financials['Date'] = snp500_financials['Report Date']

# Keep only columns that match df
cols_needed = ['Company', 'Ticker', 'Industry', 'Stage', 'Funds_Raised', 
               'Country', 'Date', 'Laid_Off_Count', 'Laid_Off_Count_Log',
               'Percentage', 'Cash_Reserves', 'Profit_Margin', 
               'Revenue_Growth', 'Laid_Off']

sp500_clean = snp500_financials[cols_needed].copy()

print(sp500_clean.shape)
print(sp500_clean.head())

(1346, 14)
    Company Ticker Industry     Stage  Funds_Raised        Country       Date  \
325     ABT    ABT  Unknown  Post-IPO           NaN  United States 2020-09-30   
326     ABT    ABT  Unknown  Post-IPO           NaN  United States 2020-12-31   
327     ABT    ABT  Unknown  Post-IPO           NaN  United States 2021-03-31   
328     ABT    ABT  Unknown  Post-IPO           NaN  United States 2021-06-30   
329     ABT    ABT  Unknown  Post-IPO           NaN  United States 2021-09-30   

     Laid_Off_Count  Laid_Off_Count_Log  Percentage  Cash_Reserves  \
325               0                   0           0   4.731000e+09   
326               0                   0           0   7.148000e+09   
327               0                   0           0   8.372000e+09   
328               0                   0           0   8.944000e+09   
329               0                   0           0   9.692000e+09   

     Profit_Margin  Revenue_Growth  Laid_Off  
325       0.139162             NaN

In [25]:
# Align columns in df to match
df_label1 = df[cols_needed].copy()

# Combine label 1 (layoffs) + label 0 (S&P 500 healthy companies)
df_final = pd.concat([df_label1, sp500_clean], axis=0, ignore_index=True)

print(f"Total rows: {df_final.shape}")
print(df_final['Laid_Off'].value_counts())

Total rows: (3735, 14)
Laid_Off
1    2389
0    1346
Name: count, dtype: int64


In [26]:
df_final.to_csv('/Users/yashwanthkumar/Desktop/Desktop/ML project/early_layoff_warning/data/master_dataset.csv', index=False)
print("Saved master_dataset.csv")
print(df_final.shape)
print(df_final.isnull().sum())

Saved master_dataset.csv
(3735, 14)
Company                  0
Ticker                1909
Industry                 0
Stage                    0
Funds_Raised          1346
Country                  0
Date                     0
Laid_Off_Count           0
Laid_Off_Count_Log       0
Percentage             701
Cash_Reserves         2131
Profit_Margin         2130
Revenue_Growth        2212
Laid_Off                 0
dtype: int64


In [ ]:
#doesnot effect the model
df_final.drop(columns=["Ticker"], inplace=True)

#companies who didnt laid off have a 0% of laid offs
df_final["Percentage"] = df_final["Percentage"].fillna(0)

#public companies doesnt raise funds
df_final["Funds_Raised"] = df_final["Funds_Raised"].fillna(0)

for col in ["Cash_Reserves", "Profit_Margin", "Revenue_Growth"]:
    median_val = df_final[col].median()

    df_final[col] = df_final[col].fillna(median_val)
    print(f"median value for {col} is {median_val}")
# Verify no nulls left
print("\nNull counts after filling:")
print(df_final.isnull().sum())
print(f"\nShape: {df_final.shape}")




median value for Cash_Reserves is 3523200000.0
median value for Profit_Margin is 0.101995318247687
median value for Revenue_Growth is 0.018416206261510082

Null counts after filling:
Company               0
Industry              0
Stage                 0
Funds_Raised          0
Country               0
Date                  0
Laid_Off_Count        0
Laid_Off_Count_Log    0
Percentage            0
Cash_Reserves         0
Profit_Margin         0
Revenue_Growth        0
Laid_Off              0
dtype: int64

Shape: (3735, 13)


In [29]:
df_final.head(10)

,Company,Industry,Stage,Funds_Raised,Country,Date,Laid_Off_Count,Laid_Off_Count_Log,Percentage,Cash_Reserves,Profit_Margin,Revenue_Growth,Laid_Off
0,Oda,Food,Unknown,691.0,Norway,2024-06-05,150.0,5.017280,0.00,3.523200e+09,0.101995,0.018416,1
1,Pagaya,Finance,Post-IPO,2000.0,Israel,2024-06-05,100.0,4.615121,0.20,3.523200e+09,0.101995,0.018416,1
2,Aleph Farms,Food,Unknown,119.0,Israel,2024-06-05,30.0,3.433987,0.30,3.523200e+09,0.101995,0.018416,1
3,MoonPay,Crypto,Unknown,651.0,United States,2024-06-05,30.0,3.433987,0.10,3.523200e+09,0.101995,0.018416,1
4,Microsoft,Other,Post-IPO,1.0,United States,2024-06-03,1000.0,6.908755,0.00,8.002100e+10,0.354667,-0.002612,1
5,OrCam,Healthcare,Unknown,86.0,Israel,2024-06-03,100.0,4.615121,0.50,3.523200e+09,0.101995,0.018416,1
6,Google,Consumer,Post-IPO,26.0,United States,2024-05-31,100.0,4.615121,0.00,1.080900e+11,0.293796,-0.066864,1
7,Tropic,Finance,Series B,67.0,United States,2024-05-31,40.0,3.713572,0.00,3.523200e+09,0.101995,0.018416,1
8,FlightStats,Travel,Acquired,3.0,United States,2024-05-30,73.0,4.304065,0.00,3.523200e+09,0.101995,0.018416,1
9,ICANN,Infrastructure,Unknown,172.5,United States,2024-05-30,33.0,3.526361,0.07,3.523200e+09,0.101995,0.018416,1
